In [1]:
import pandas as pd
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.series import DataPoint

# Load data
metrics = pd.read_csv("../data/metrics_summary.csv", index_col=0)
monthly = pd.read_csv("../data/query1_monthly_avg.csv")
ranking = pd.read_csv("../data/query3_asset_ranking.csv")

# Create workbook
wb = Workbook()

# Remove default sheet
wb.remove(wb.active)

# Colors
dark_blue = "1F3864"
light_blue = "BDD7EE"
white = "FFFFFF"
green = "70AD47"
red = "FF0000"
yellow = "FFD700"

print("Workbook created, building sheets...")

Workbook created, building sheets...


In [2]:
ws1 = wb.create_sheet("Summary Dashboard")

# Title
ws1["B2"] = "Financial Portfolio Analysis"
ws1["B2"].font = Font(bold=True, size=18, color=white)
ws1["B2"].fill = PatternFill("solid", fgColor=dark_blue)
ws1.merge_cells("B2:H2")
ws1["B2"].alignment = Alignment(horizontal="center")

# Subtitle
ws1["B3"] = "Risk and Return Analysis | 2021 to 2025"
ws1["B3"].font = Font(italic=True, size=11, color=dark_blue)
ws1.merge_cells("B3:H3")

# Headers
headers = ["Asset", "Annual Return", "Volatility", "Sharpe Ratio", "Max Drawdown", "Rank"]
header_cols = ["B", "C", "D", "E", "F", "G"]

for col, header in zip(header_cols, headers):
    cell = ws1[f"{col}5"]
    cell.value = header
    cell.font = Font(bold=True, color=white)
    cell.fill = PatternFill("solid", fgColor=dark_blue)
    cell.alignment = Alignment(horizontal="center")

# Data rows
assets = ranking["index"].tolist() if "index" in ranking.columns else ranking.iloc[:, 0].tolist()
row = 6
for i, asset_row in ranking.iterrows():
    values = [
        asset_row.iloc[0],
        f"{asset_row['Annualized Return']*100:.1f}%",
        f"{asset_row['Annualized Volatility']*100:.1f}%",
        f"{asset_row['Sharpe Ratio']:.2f}",
        f"{asset_row['Max Drawdown']*100:.1f}%",
        int(asset_row['Rank'])
    ]
    for col, val in zip(header_cols, values):
        cell = ws1[f"{col}{row}"]
        cell.value = val
        cell.alignment = Alignment(horizontal="center")
        if row % 2 == 0:
            cell.fill = PatternFill("solid", fgColor=light_blue)
    row += 1

# Column widths
ws1.column_dimensions["B"].width = 12
ws1.column_dimensions["C"].width = 16
ws1.column_dimensions["D"].width = 14
ws1.column_dimensions["E"].width = 14
ws1.column_dimensions["F"].width = 15
ws1.column_dimensions["G"].width = 8

print("Sheet 1 done.")

Sheet 1 done.


In [3]:
# Sheet 2 - Monthly average prices
ws2 = wb.create_sheet("Monthly Prices")

ws2["A1"] = "Monthly Average Prices by Asset"
ws2["A1"].font = Font(bold=True, size=13, color=white)
ws2["A1"].fill = PatternFill("solid", fgColor=dark_blue)
ws2.merge_cells("A1:F1")

for row in dataframe_to_rows(monthly, index=False, header=True):
    ws2.append(row)

for col in ws2.iter_cols(min_row=2, max_row=2):
    for cell in col:
        cell.font = Font(bold=True, color=white)
        cell.fill = PatternFill("solid", fgColor=dark_blue)

for col_letter in ["A", "B", "C", "D", "E", "F"]:
    ws2.column_dimensions[col_letter].width = 14

# Sheet 3 - Full metrics
ws3 = wb.create_sheet("Risk Metrics")

ws3["A1"] = "Risk and Return Metrics"
ws3["A1"].font = Font(bold=True, size=13, color=white)
ws3["A1"].fill = PatternFill("solid", fgColor=dark_blue)
ws3.merge_cells("A1:E1")

for row in dataframe_to_rows(metrics.reset_index(), index=False, header=True):
    ws3.append(row)

for col in ws3.iter_cols(min_row=2, max_row=2):
    for cell in col:
        cell.font = Font(bold=True, color=white)
        cell.fill = PatternFill("solid", fgColor=dark_blue)

for col_letter in ["A", "B", "C", "D", "E"]:
    ws3.column_dimensions[col_letter].width = 22

print("Sheet 2 and 3 done.")

Sheet 2 and 3 done.


In [5]:
import os

os.makedirs("../dashboard", exist_ok=True)

output_path = "../dashboard/portfolio_dashboard.xlsx"
wb.save(output_path)
print("Dashboard saved to dashboard/portfolio_dashboard.xlsx")

Dashboard saved to dashboard/portfolio_dashboard.xlsx
